# TTS 모듈 테스트

이 노트북은 `tts/generate_tts.py` 스크립트의 기능을 테스트합니다.
1. 필요한 라이브러리와 모듈을 임포트합니다.
2. 테스트용 입력 텍스트 파일을 생성합니다.
3. TTS 설정을 로드합니다.
4. TTS 생성을 실행합니다.
5. 생성된 오디오 파일을 확인하고 재생합니다.

In [1]:
import os
import sys
import glob
from IPython.display import Audio, display

# 프로젝트 루트를 sys.path에 추가하여 모듈 임포트
project_root = os.path.abspath('.') # 노트북이 프로젝트 루트에 있다고 가정
if project_root not in sys.path:
    sys.path.append(project_root)

from tts.generate_tts import process_text_file
from utils.file_utils import read_yaml, ensure_dir_exists
from utils.logging_utils import setup_logger

logger = setup_logger('tts_test_notebook')

## 1. 설정 및 경로 정의

In [2]:
config_path = 'tts/tts_config.yaml'
test_input_dir = 'data/raw'
test_input_filename = 'test_tts_input.txt'
test_input_path = os.path.join(test_input_dir, test_input_filename)
output_dir = 'data/tts_test_output' # 테스트용 별도 출력 디렉토리

# 필요한 디렉토리 생성
ensure_dir_exists(test_input_dir)
ensure_dir_exists(output_dir)

print(f"Config Path: {config_path}")
print(f"Test Input Path: {test_input_path}")
print(f"Output Directory: {output_dir}")

Created directory: data/tts_test_output
Config Path: tts/tts_config.yaml
Test Input Path: data/raw/test_tts_input.txt
Output Directory: data/tts_test_output


## 2. 테스트용 입력 텍스트 파일 생성

In [3]:
sample_text = [
    "안녕하세요, 루시퍼 프로젝트입니다.",
    "이것은 TTS 기능 테스트입니다.",
    "한국어 음성 합성이 잘 되는지 확인합니다."
]

try:
    with open(test_input_path, 'w', encoding='utf-8') as f:
        for line in sample_text:
            f.write(line + '\n')
    logger.info(f"Created sample input file: {test_input_path}")
except Exception as e:
    logger.error(f"Failed to create sample input file: {e}")

2025-04-16 08:12:18,453 - tts_test_notebook - INFO - Created sample input file: data/raw/test_tts_input.txt


## 3. TTS 설정 로드

In [4]:
config = read_yaml(config_path)
if config:
    logger.info(f"Loaded TTS configuration: {config}")
else:
    logger.error("Failed to load TTS configuration.")

2025-04-16 08:12:31,211 - tts_test_notebook - INFO - Loaded TTS configuration: {'language': 'ko', 'engine': 'gtts', 'output_format': 'mp3'}


## 4. TTS 생성 실행

In [5]:
if config:
    logger.info(f"Starting TTS generation from {test_input_path} to {output_dir}")
    process_text_file(test_input_path, output_dir, config)
    logger.info("TTS generation finished.")
else:
    logger.warning("Skipping TTS generation due to missing config.")

2025-04-16 08:12:36,189 - tts_test_notebook - INFO - Starting TTS generation from data/raw/test_tts_input.txt to data/tts_test_output
2025-04-16 08:12:36,375 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0001.mp3
2025-04-16 08:12:36,375 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0001.mp3
2025-04-16 08:12:36,477 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0002.mp3
2025-04-16 08:12:36,477 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0002.mp3
2025-04-16 08:12:36,587 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0003.mp3
2025-04-16 08:12:36,587 - tts_test_notebook - INFO - TTS generation finished.
2025-04-16 08:12:36,587 - tts.generate_tts - INFO - Successfully generated TTS audio at data/tts_test_output/line_0003.mp3
2025-04-16 08:12:36,587 - tts_test_notebook - INFO

## 5. 생성된 오디오 파일 확인 및 재생

In [6]:
# 생성된 파일 목록 확인 (gTTS는 mp3로 저장)
output_format = config.get('output_format', 'mp3')
if config.get('engine', 'gtts') == 'gtts':
    output_format = 'mp3' # gTTS는 mp3만 지원

generated_files = glob.glob(os.path.join(output_dir, f"*.{output_format}"))

if generated_files:
    logger.info(f"Generated {len(generated_files)} audio files:")
    for f in sorted(generated_files):
        print(f" - {os.path.basename(f)}")
    
    # 첫 번째 생성된 파일 재생
    first_audio_file = sorted(generated_files)[0]
    print(f"\nPlaying first generated file: {os.path.basename(first_audio_file)}")
    display(Audio(first_audio_file))
else:
    logger.warning(f"No audio files found in {output_dir} with format {output_format}.")

2025-04-16 08:12:50,448 - tts_test_notebook - INFO - Generated 3 audio files:
 - line_0001.mp3
 - line_0002.mp3
 - line_0003.mp3

Playing first generated file: line_0001.mp3
 - line_0001.mp3
 - line_0002.mp3
 - line_0003.mp3

Playing first generated file: line_0001.mp3
